# 第5章 高度な深層学習 ― サンプルコード
『AI実装検定 A級・S級 完全版』の第5章のリストを、本文と同じ順に実行できます。

**使い方**：上のメニュー［ランタイム］→［すべてのセルを実行］。CPU のランタイムで動きます（GPU は不要）。

**本書の動作確認環境**（Python 3.13.17）：NumPy 2.5.3、PyTorch 2.14.1、TensorFlow 2.21.0、Keras 3.15.1、Transformers 5.20.2

In [ ]:
# 版をそろえたいとき（任意）：本書と同じ版を入れます。通常は Colab の標準版でそのまま動きます。
# !pip install -q numpy==2.5.3 torch==2.14.1 tensorflow-cpu==2.21.0 keras==3.15.1 transformers==5.20.2
import sys; print(sys.version)


## リスト5.1　パラメータ数の比較と受容野の計算

In [ ]:
C = 64
p5 = 5 * 5 * C * C              # 5x5 を 1 層
p33 = 2 * (3 * 3 * C * C)       # 3x3 を 2 層（同じ 5x5 の範囲を見る）
print(p5, p33, round(p33 / p5, 2))

def receptive_field(layers):
    r, j = 1, 1                 # 受容野と、出力 1 マスあたりの入力の間隔
    for k, s in layers:
        r += (k - 1) * j
        j *= s
    return r

print(receptive_field([(3, 1), (3, 1)]))
print(receptive_field([(3, 1)] * 3))
print(receptive_field([(5, 1), (2, 2), (5, 1), (2, 2)]))   # LeNet の畳み込み 2 回

**本書に載っている出力**
```
102400 73728 0.72
5
7
16
```

## リスト5.2　LeNet-5 の各層とパラメータ数

In [ ]:
import torch
from torch import nn

net = nn.Sequential(
    nn.Conv2d(1, 6, 5), nn.ReLU(), nn.MaxPool2d(2),     # 32 -> 28 -> 14
    nn.Conv2d(6, 16, 5), nn.ReLU(), nn.MaxPool2d(2),    # 14 -> 10 -> 5
    nn.Flatten(),
    nn.Linear(16 * 5 * 5, 120), nn.ReLU(),
    nn.Linear(120, 84), nn.ReLU(),
    nn.Linear(84, 10),
)
x = torch.zeros(1, 1, 32, 32)
for layer in net:
    x = layer(x)
    n = sum(p.numel() for p in layer.parameters())
    if n or isinstance(layer, (nn.MaxPool2d, nn.Flatten)):
        print(f"{layer.__class__.__name__:10s} {tuple(x.shape)}  params={n}")
print(sum(p.numel() for p in net.parameters()))

**本書に載っている出力**
```
Conv2d     (1, 6, 28, 28)  params=156
MaxPool2d  (1, 6, 14, 14)  params=0
Conv2d     (1, 16, 10, 10)  params=2416
MaxPool2d  (1, 16, 5, 5)  params=0
Flatten    (1, 400)  params=0
Linear     (1, 120)  params=48120
Linear     (1, 84)  params=10164
Linear     (1, 10)  params=850
61706
```

## リスト5.3　残差ブロックの実装

In [ ]:
import torch
from torch import nn

class BasicBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.conv1 = nn.Conv2d(c, c, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(c)
        self.conv2 = nn.Conv2d(c, c, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(c)

    def forward(self, x):
        out = torch.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return torch.relu(out + x)          # F(x) + x：スキップ接続

blk = BasicBlock(4).eval()
x = torch.randn(1, 4, 8, 8)
print(blk(x).shape)
with torch.no_grad():                       # 畳み込みの重みを 0 にすると F(x) = 0
    blk.conv1.weight.zero_()
    blk.conv2.weight.zero_()
print(torch.allclose(blk(x), torch.relu(x), atol=1e-6))   # 出力は x そのまま（ReLU 後）

**本書に載っている出力**
```
torch.Size([1, 4, 8, 8])
True
```

## リスト5.4　20 層での入力側の勾配の大きさ

In [ ]:
import torch
from torch import nn

torch.manual_seed(0)
depth, d = 20, 16

def input_grad(residual):
    layers = [nn.Linear(d, d) for _ in range(depth)]
    x = torch.randn(1, d, requires_grad=True)
    h = x
    for lin in layers:
        h = h + torch.sigmoid(lin(h)) if residual else torch.sigmoid(lin(h))
    h.sum().backward()
    return x.grad.norm().item()

print("%.2e" % input_grad(False))      # 20 層をそのまま重ねる
print("%.2e" % input_grad(True))       # 各層に x + f(x) の近道をつける

**本書に載っている出力**
```
1.41e-17
4.18e+00
```

## リスト5.5　チャンネル 256 での通常ブロックとボトルネックの比較

In [ ]:
from torch import nn

def n(*layers):
    return sum(p.numel() for m in layers for p in m.parameters())

plain = (nn.Conv2d(256, 256, 3, padding=1, bias=False), nn.Conv2d(256, 256, 3, padding=1, bias=False))
bott = (nn.Conv2d(256, 64, 1, bias=False), nn.Conv2d(64, 64, 3, padding=1, bias=False),
        nn.Conv2d(64, 256, 1, bias=False))
print(n(*plain), n(*bott))

**本書に載っている出力**
```
1179648 69632
```

## リスト5.6　表（cfg）から VGG-16 を組み立て、パラメータ数を数える

In [ ]:
import torch
from torch import nn

# 論文の表（VGG-16）：数字 = 3x3 畳み込みの出力チャンネル数、"M" = 2x2 最大プーリング
cfg = [64, 64, "M", 128, 128, "M", 256, 256, 256, "M",
       512, 512, 512, "M", 512, 512, 512, "M"]

def make_features(cfg):
    layers, c_in = [], 3
    for v in cfg:
        if v == "M":
            layers.append(nn.MaxPool2d(2))
        else:
            layers += [nn.Conv2d(c_in, v, 3, padding=1), nn.ReLU()]
            c_in = v
    return nn.Sequential(*layers)

features = make_features(cfg)
classifier = nn.Sequential(
    nn.Flatten(),
    nn.Linear(512 * 7 * 7, 4096), nn.ReLU(), nn.Dropout(),
    nn.Linear(4096, 4096), nn.ReLU(), nn.Dropout(),
    nn.Linear(4096, 1000),
)
count = lambda m: sum(p.numel() for p in m.parameters())
print(features(torch.zeros(1, 3, 224, 224)).shape)
print(count(features), count(classifier), count(features) + count(classifier))
print(round(count(classifier) / (count(features) + count(classifier)), 3))

**本書に載っている出力**
```
torch.Size([1, 512, 7, 7])
14714688 123642856 138357544
0.894
```

## リスト5.7　Inception モジュール（3a）の実装と、1×1 による節約

In [ ]:
import torch
from torch import nn

class Inception(nn.Module):
    def __init__(self, c_in, c1, c3r, c3, c5r, c5, cp):
        super().__init__()
        self.b1 = nn.Conv2d(c_in, c1, 1)                                        # 1x1
        self.b2 = nn.Sequential(nn.Conv2d(c_in, c3r, 1), nn.ReLU(),
                                nn.Conv2d(c3r, c3, 3, padding=1))               # 1x1 -> 3x3
        self.b3 = nn.Sequential(nn.Conv2d(c_in, c5r, 1), nn.ReLU(),
                                nn.Conv2d(c5r, c5, 5, padding=2))               # 1x1 -> 5x5
        self.b4 = nn.Sequential(nn.MaxPool2d(3, 1, 1), nn.Conv2d(c_in, cp, 1))  # pool -> 1x1
    def forward(self, x):
        return torch.cat([self.b1(x), self.b2(x), self.b3(x), self.b4(x)], dim=1)

count = lambda m: sum(p.numel() for p in m.parameters())
m = Inception(192, 64, 96, 128, 16, 32, 32)       # GoogLeNet の inception (3a)
print(m(torch.zeros(1, 192, 28, 28)).shape)
for name in ["b1", "b2", "b3", "b4"]:
    print(name, count(getattr(m, name)))
print("total", count(m))

direct5 = 192 * 32 * 5 * 5 + 32                   # 1x1 で絞らず、5x5 を直接かける場合
reduced5 = count(m.b3)
print(direct5, reduced5, round(reduced5 / direct5, 3))
hw = 28 * 28                                      # 掛け算の回数（バイアスを除く）
print(192 * 32 * 25 * hw, (192 * 16 + 16 * 32 * 25) * hw)

**本書に載っている出力**
```
torch.Size([1, 256, 28, 28])
b1 12352
b2 129248
b3 15920
b4 6176
total 163696
153632 15920 0.104
120422400 12443648
```

## リスト5.8　WideResNet の実装と、幅を変えたときのパラメータ数

In [ ]:
from torch import nn

class Block(nn.Module):                            # BN-ReLU-Conv を 2 回（pre-activation）
    def __init__(self, c_in, c_out, stride):
        super().__init__()
        self.bn1, self.bn2 = nn.BatchNorm2d(c_in), nn.BatchNorm2d(c_out)
        self.c1 = nn.Conv2d(c_in, c_out, 3, stride, 1, bias=False)
        self.c2 = nn.Conv2d(c_out, c_out, 3, 1, 1, bias=False)
        self.short = None if (c_in == c_out and stride == 1) else nn.Conv2d(c_in, c_out, 1, stride, bias=False)
    def forward(self, x):
        o = nn.functional.relu(self.bn1(x))
        y = self.c2(nn.functional.relu(self.bn2(self.c1(o))))
        return y + (x if self.short is None else self.short(o))

def wide_resnet(depth, k, classes=10):
    n = (depth - 4) // 6                           # 各グループのブロック数
    widths = [16, 16 * k, 32 * k, 64 * k]
    layers, c = [nn.Conv2d(3, widths[0], 3, 1, 1, bias=False)], widths[0]
    for g, w in enumerate(widths[1:]):
        for i in range(n):
            layers.append(Block(c, w, 2 if (i == 0 and g > 0) else 1)); c = w
    layers += [nn.BatchNorm2d(c), nn.ReLU(), nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(c, classes)]
    return nn.Sequential(*layers)

count = lambda m: sum(p.numel() for p in m.parameters())
for depth, k in [(16, 1), (16, 2), (16, 4), (16, 8), (28, 10)]:
    print(f"WRN-{depth}-{k}: {count(wide_resnet(depth, k)):,}")
print(round(count(wide_resnet(16, 4)) / count(wide_resnet(16, 2)), 2))

**本書に載っている出力**
```
WRN-16-1: 175,066
WRN-16-2: 691,674
WRN-16-4: 2,748,890
WRN-16-8: 10,961,370
WRN-28-10: 36,479,194
3.97
```

## リスト5.9　Dense ブロック（6 層、$k_0 = 64$、$k = 32$）と遷移層

In [ ]:
import torch
from torch import nn

class DenseLayer(nn.Module):
    def __init__(self, c_in, k):                   # k = 成長率（growth rate）
        super().__init__()
        self.f = nn.Sequential(nn.BatchNorm2d(c_in), nn.ReLU(), nn.Conv2d(c_in, 4 * k, 1, bias=False),
                               nn.BatchNorm2d(4 * k), nn.ReLU(), nn.Conv2d(4 * k, k, 3, padding=1, bias=False))
    def forward(self, x):
        return torch.cat([x, self.f(x)], dim=1)    # 入力に、新しい k チャンネルを「連結」する

k0, k, L = 64, 32, 6
block = nn.Sequential(*[DenseLayer(k0 + i * k, k) for i in range(L)])
x = torch.zeros(1, k0, 28, 28)
for i, layer in enumerate(block, 1):
    x = layer(x)
    print(f"層{i}: チャンネル数 {x.shape[1]}")
trans = nn.Sequential(nn.BatchNorm2d(x.shape[1]), nn.ReLU(),
                      nn.Conv2d(x.shape[1], x.shape[1] // 2, 1, bias=False), nn.AvgPool2d(2))   # 圧縮率 0.5
y = trans(x)
print(tuple(y.shape))
print("直接の接続の数:", L * (L + 1) // 2)
print(sum(p.numel() for p in block.parameters()), sum(p.numel() for p in trans.parameters()))

**本書に載っている出力**
```
層1: チャンネル数 96
層2: チャンネル数 128
層3: チャンネル数 160
層4: チャンネル数 192
層5: チャンネル数 224
層6: チャンネル数 256
(1, 128, 14, 14)
直接の接続の数: 21
335040 33280
```

## リスト5.10　depthwise separable 畳み込みと、MobileNetV2 の Inverted Residual

In [ ]:
import torch
from torch import nn

M, N, Dk = 64, 128, 3                              # 入力・出力チャンネル数とフィルタの大きさ
std = nn.Conv2d(M, N, Dk, padding=1, bias=False)
dw = nn.Conv2d(M, M, Dk, padding=1, groups=M, bias=False)    # depthwise：チャンネルごとに独立
pw = nn.Conv2d(M, N, 1, bias=False)                          # pointwise：1x1 でチャンネルを混ぜる
count = lambda *ms: sum(p.numel() for m in ms for p in m.parameters())
print(count(std), count(dw), count(pw), count(dw, pw))
print(round(count(dw, pw) / count(std), 4), round(1 / N + 1 / Dk ** 2, 4))

x = torch.zeros(1, M, 56, 56)
print(std(x).shape, pw(dw(x)).shape)

class InvertedResidual(nn.Module):                 # MobileNetV2：広げて → depthwise → 絞る
    def __init__(self, c, t=6):
        super().__init__()
        self.f = nn.Sequential(
            nn.Conv2d(c, c * t, 1, bias=False), nn.BatchNorm2d(c * t), nn.ReLU6(),
            nn.Conv2d(c * t, c * t, 3, padding=1, groups=c * t, bias=False), nn.BatchNorm2d(c * t), nn.ReLU6(),
            nn.Conv2d(c * t, c, 1, bias=False), nn.BatchNorm2d(c))                  # 最後は活性化なし
    def forward(self, x):
        return x + self.f(x)

blk = InvertedResidual(24)
print(blk(torch.zeros(1, 24, 56, 56)).shape, count(blk))

**本書に載っている出力**
```
73728 576 8192 8768
0.1189 0.1189
torch.Size([1, 128, 56, 56]) torch.Size([1, 128, 56, 56])
torch.Size([1, 24, 56, 56]) 8832
```

## リスト5.11　複合スケーリング：φ を増やしたときの深さ・幅・解像度・計算量

In [ ]:
alpha, beta, gamma = 1.2, 1.1, 1.15                 # 論文の方針：α·β²·γ² ≈ 2
print(round(alpha * beta ** 2 * gamma ** 2, 3))
print("phi  深さ  幅    解像度  計算量(倍)")
for phi in range(5):
    d, w, r = alpha ** phi, beta ** phi, gamma ** phi
    print(f"{phi:>3}  {d:.2f}  {w:.2f}  {r:.2f}   {d * w ** 2 * r ** 2:.2f}")

C, r = 96, 4                                       # Squeeze-and-Excitation（SE）ブロック
se = (C * (C // r) + C // r) + ((C // r) * C + C)  # 絞る全結合 ＋ 戻す全結合
print(se)

**本書に載っている出力**
```
1.92
phi  深さ  幅    解像度  計算量(倍)
  0  1.00  1.00  1.00   1.00
  1  1.20  1.10  1.15   1.92
  2  1.44  1.21  1.32   3.69
  3  1.73  1.33  1.52   7.08
  4  2.07  1.46  1.75   13.60
4728
```

## リスト5.12　Keras 版の Inception モジュールと depthwise separable 畳み込み

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"           # TensorFlow のログを減らす
import keras
from keras import layers

def inception(x, c1, c3r, c3, c5r, c5, cp):
    b1 = layers.Conv2D(c1, 1)(x)
    b2 = layers.Conv2D(c3, 3, padding="same")(layers.Conv2D(c3r, 1, activation="relu")(x))
    b3 = layers.Conv2D(c5, 5, padding="same")(layers.Conv2D(c5r, 1, activation="relu")(x))
    b4 = layers.Conv2D(cp, 1)(layers.MaxPooling2D(3, 1, padding="same")(x))
    return layers.Concatenate()([b1, b2, b3, b4])

inp = keras.Input((28, 28, 192))                   # Keras は (高さ, 幅, チャンネル) の順
m = keras.Model(inp, inception(inp, 64, 96, 128, 16, 32, 32))
print(m.output_shape, m.count_params())            # PyTorch 版と同じ 163,696

inp = keras.Input((56, 56, 64))
dw = layers.DepthwiseConv2D(3, padding="same", use_bias=False)(inp)
pw = layers.Conv2D(128, 1, use_bias=False)(dw)
m2 = keras.Model(inp, pw)
print(m2.output_shape, m2.count_params())          # 576 + 8192 = 8768

**本書に載っている出力**
```
(None, 28, 28, 256) 163696
(None, 56, 56, 128) 8768
```

## リスト5.13　LSTMCell を手で再現（PyTorch の並びは i, f, g, o）

In [ ]:
import torch
from torch import nn

torch.manual_seed(0)
cell = nn.LSTMCell(3, 4)
x, h, c = torch.randn(1, 3), torch.randn(1, 4), torch.randn(1, 4)

z = x @ cell.weight_ih.T + cell.bias_ih + h @ cell.weight_hh.T + cell.bias_hh
i, f, g, o = z.chunk(4, dim=1)                       # PyTorch の並び順：i, f, g, o
i, f, o, g = torch.sigmoid(i), torch.sigmoid(f), torch.sigmoid(o), torch.tanh(g)
c_new = f * c + i * g
h_new = o * torch.tanh(c_new)
h_ref, c_ref = cell(x, (h, c))
print(torch.allclose(h_new, h_ref, atol=1e-6), torch.allclose(c_new, c_ref, atol=1e-6))
print(cell.weight_ih.shape, cell.weight_hh.shape)    # ゲート 4 つ分がまとまっている

**本書に載っている出力**
```
True True
torch.Size([16, 3]) torch.Size([16, 4])
```

## リスト5.14　GRUCell を手で再現（PyTorch の並びは r, z, n）

In [ ]:
import torch
from torch import nn

torch.manual_seed(0)
cell = nn.GRUCell(3, 4)
x, h = torch.randn(1, 3), torch.randn(1, 4)

gi = x @ cell.weight_ih.T + cell.bias_ih
gh = h @ cell.weight_hh.T + cell.bias_hh
i_r, i_z, i_n = gi.chunk(3, dim=1)                   # PyTorch の並び順：r, z, n
h_r, h_z, h_n = gh.chunk(3, dim=1)
r = torch.sigmoid(i_r + h_r)                         # リセットゲート
z = torch.sigmoid(i_z + h_z)                         # 更新ゲート
n = torch.tanh(i_n + r * h_n)                        # 候補
h_new = (1 - z) * n + z * h
print(torch.allclose(h_new, cell(x, h), atol=1e-6))
print(cell.weight_ih.shape)                          # ゲート 3 つ分

**本書に載っている出力**
```
True
torch.Size([12, 3])
```

## リスト5.15　双方向・2 層の LSTM

In [ ]:
import torch
from torch import nn

lstm = nn.LSTM(input_size=8, hidden_size=16, num_layers=2, batch_first=True, bidirectional=True)
x = torch.randn(4, 10, 8)                            # (バッチ, 時刻, 特徴)
out, (h, c) = lstm(x)
print(out.shape)                                     # 前向きと後ろ向きがつながる：2 × 16
print(h.shape)                                       # (層数 × 方向, バッチ, 隠れ次元)

**本書に載っている出力**
```
torch.Size([4, 10, 32])
torch.Size([4, 4, 16])
```

## リスト5.16　Seq2Seq の形と注意の計算（形の確認）

In [ ]:
import torch
from torch import nn

N, T, V, E, H = 2, 7, 100, 8, 16
emb = nn.Embedding(V, E)
enc = nn.GRU(E, H, batch_first=True)
tokens = torch.randint(0, V, (N, T))
enc_out, enc_h = enc(emb(tokens))
print(enc_out.shape, enc_h.shape)                     # 全時刻の状態 / 最後の状態（文の要約）

dec = nn.GRU(E, H, batch_first=True)
y_prev = torch.randint(0, V, (N, 1))                  # 直前に出力した単語
dec_out, dec_h = dec(emb(y_prev), enc_h)              # エンコーダの要約から開始
scores = (enc_out * dec_h.transpose(0, 1)).sum(-1)    # 注意：各時刻との内積
weights = torch.softmax(scores, dim=1)
context = (weights.unsqueeze(-1) * enc_out).sum(1)
print(scores.shape, weights.sum(dim=1).round(decimals=4).tolist(), context.shape)

**本書に載っている出力**
```
torch.Size([2, 7, 16]) torch.Size([1, 2, 16])
torch.Size([2, 7]) [1.0, 1.0] torch.Size([2, 16])
```

## リスト5.17　Attention の手計算を NumPy で確かめる

In [ ]:
import numpy as np

h = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])   # エンコーダの 3 時刻の状態
s = np.array([1.0, 0.0])                             # デコーダの現在の状態
scores = h @ s                                       # 内積で似ている度合いを測る
w = np.exp(scores) / np.exp(scores).sum()            # softmax
context = w @ h                                      # 重み付き和
print(scores)
print(w.round(4), w.sum())
print(context.round(4))

**本書に載っている出力**
```
[1. 0. 1.]
[0.4223 0.1554 0.4223] 1.0
[0.8446 0.5777]
```

## リスト5.18　HRED の形の確認（発話 3 つ、各 5 語）と GRU のパラメータ数

In [ ]:
import torch
from torch import nn

V, E, Hu, Hc = 100, 8, 16, 24                      # 語彙・埋め込み・発話用 GRU・文脈用 GRU の次元
emb = nn.Embedding(V, E)
utt_enc = nn.GRU(E, Hu, batch_first=True)          # 1 つの発話（語の列）を 1 本のベクトルにする
ctx_enc = nn.GRU(Hu, Hc, batch_first=True)         # 発話ベクトルの列から「会話の文脈」を作る
dec = nn.GRU(E, Hc, batch_first=True)              # 文脈から次の発話を生成する
out = nn.Linear(Hc, V)

turns, words = 3, 5                                # 過去の 3 発話、各 5 語
dialog = torch.randint(0, V, (turns, words))
_, h = utt_enc(emb(dialog))                        # h: (1, 3, 16) = 発話ごとの 16 次元ベクトル
_, c = ctx_enc(h)                                  # h を「3 発話の 1 つの会話」として読む → c: (1, 1, 24)
reply = torch.randint(0, V, (1, 4))
y, _ = dec(emb(reply), c)                          # 文脈を初期状態にして返答を生成
print(h.shape, c.shape, out(y).shape)
count = lambda m: sum(p.numel() for p in m.parameters())
print(count(utt_enc), count(ctx_enc), count(dec))
print(3 * (Hu * E + Hu * Hu + 2 * Hu))             # GRU のパラメータ数の式

**本書に載っている出力**
```
torch.Size([1, 3, 16]) torch.Size([1, 1, 24]) torch.Size([1, 4, 100])
1248 3024 2448
1248
```

## リスト5.19　clip_grad_norm_ の効果

In [ ]:
import torch
from torch import nn

torch.manual_seed(0)
rnn = nn.RNN(4, 8, batch_first=True)
with torch.no_grad():
    for p in rnn.parameters():
        p.mul_(4.0)                                  # 重みを大きくして勾配爆発を起こしやすくする
x = torch.randn(2, 50, 4)
out, _ = rnn(x)
out.pow(2).sum().backward()
before = torch.nn.utils.clip_grad_norm_(rnn.parameters(), max_norm=1.0)   # 戻り値はクリップ前のノルム
after = torch.sqrt(sum((p.grad ** 2).sum() for p in rnn.parameters()))
print(before.item() > 1.0, round(after.item(), 4))

**本書に載っている出力**
```
True 1.0
```

## リスト5.20　Scaled Dot-Product Attention の手計算と PyTorch の一致

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F

X = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])   # 3 トークン、次元 2
Q = K = V = X                                        # 重みを単位行列にして手計算しやすくする
scores = (Q @ K.T) / np.sqrt(2)
e = np.exp(scores - scores.max(axis=1, keepdims=True))
A = e / e.sum(axis=1, keepdims=True)                 # 行ごとの softmax
print(scores.round(4))
print(A.round(4))
print((A @ V).round(4))
t = torch.tensor(X, dtype=torch.float32)
ref = F.scaled_dot_product_attention(t[None], t[None], t[None])[0]
print(np.allclose(A @ V, ref.numpy(), atol=1e-5))    # PyTorch の関数と一致

**本書に載っている出力**
```
[[0.7071 0.     0.7071]
 [0.     0.7071 0.7071]
 [0.7071 0.7071 1.4142]]
[[0.4011 0.1978 0.4011]
 [0.1978 0.4011 0.4011]
 [0.2483 0.2483 0.5035]]
[[0.8022 0.5989]
 [0.5989 0.8022]
 [0.7517 0.7517]]
True
```

## リスト5.21　√d で割る効果（d = 64）

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
d = 64
q = rng.normal(size=(100000, d))
k = rng.normal(size=(100000, d))
dots = (q * k).sum(axis=1)
print(round(dots.std(), 2), round((dots / np.sqrt(d)).std(), 2))

def max_prob(scale):
    s = rng.normal(size=(2000, 10)) * scale          # 10 トークンへのスコア
    p = np.exp(s - s.max(axis=1, keepdims=True))
    p /= p.sum(axis=1, keepdims=True)
    return round(p.max(axis=1).mean(), 2)

print(max_prob(np.sqrt(d)), max_prob(1.0))           # 割らない場合 / 割った場合の最大確率

**本書に載っている出力**
```
8.0 1.0
0.88 0.32
```

## リスト5.22　nn.MultiheadAttention（d = 512、8 ヘッド）

In [ ]:
import torch
from torch import nn

mha = nn.MultiheadAttention(embed_dim=512, num_heads=8, batch_first=True)
x = torch.randn(2, 10, 512)                          # (バッチ, トークン数, 次元)
out, w = mha(x, x, x)                                # 自己注意：Q = K = V = x
print(out.shape, w.shape)                            # w は 8 ヘッドの平均
_, w_all = mha(x, x, x, average_attn_weights=False)
print(w_all.shape)
print(sum(p.numel() for p in mha.parameters()), 4 * 512 * 512 + 4 * 512)
print(512 // 8)                                      # 1 ヘッドの次元

**本書に載っている出力**
```
torch.Size([2, 10, 512]) torch.Size([2, 10, 10])
torch.Size([2, 8, 10, 10])
1050624 1050624
64
```

## リスト5.23　位置エンコーディングと、順序の入れ替えテスト

In [ ]:
import numpy as np
import torch
from torch import nn

def positional_encoding(n, d):
    pos = np.arange(n)[:, None]
    i = np.arange(0, d, 2)[None, :]
    angle = pos / (10000 ** (i / d))
    pe = np.zeros((n, d))
    pe[:, 0::2] = np.sin(angle)
    pe[:, 1::2] = np.cos(angle)
    return pe

print(positional_encoding(4, 4).round(4))

torch.manual_seed(0)
layer = nn.TransformerEncoderLayer(16, 2, 32, dropout=0.0, batch_first=True).eval()
x = torch.randn(1, 5, 16)
perm = [3, 1, 4, 0, 2]
with torch.no_grad():
    a = layer(x)[:, perm]
    b = layer(x[:, perm])
    print(torch.allclose(a, b, atol=1e-5))           # 位置情報なし：順序を入れ替えても出力は同じ入れ替わり
    pe = torch.tensor(positional_encoding(5, 16), dtype=torch.float32)[None]
    c = layer(x + pe)[:, perm]
    d = layer(x[:, perm] + pe)
    print(torch.allclose(c, d, atol=1e-5))           # 位置情報あり：もう一致しない

**本書に載っている出力**
```
[[ 0.      1.      0.      1.    ]
 [ 0.8415  0.5403  0.01    1.    ]
 [ 0.9093 -0.4161  0.02    0.9998]
 [ 0.1411 -0.99    0.03    0.9996]]
True
False
```

## リスト5.24　エンコーダ層のパラメータ数（d = 512、FFN = 2048）

In [ ]:
from torch import nn

layer = nn.TransformerEncoderLayer(d_model=512, nhead=8, dim_feedforward=2048)
total = sum(p.numel() for p in layer.parameters())
attn = 4 * 512 * 512 + 4 * 512                  # Q, K, V, 出力の 4 つの射影
ffn = (512 * 2048 + 2048) + (2048 * 512 + 512)  # 2 つの全結合
norm = 2 * (2 * 512)                            # LayerNorm 2 つ（重みとバイアス）
print(total, attn + ffn + norm)

**本書に載っている出力**
```
3152384 3152384
```

## リスト5.25　因果マスクと softmax

In [ ]:
import torch

n = 4
mask = torch.tril(torch.ones(n, n))                  # 下三角：自分より未来は見えない
print(mask.int())
scores = torch.zeros(n, n)                           # 全スコア同じにして、マスクの効果だけを見る
scores = scores.masked_fill(mask == 0, float("-inf"))
print(torch.softmax(scores, dim=-1).round(decimals=4))

**本書に載っている出力**
```
tensor([[1, 0, 0, 0],
        [1, 1, 0, 0],
        [1, 1, 1, 0],
        [1, 1, 1, 1]], dtype=torch.int32)
tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500]])
```

## リスト5.26　マスクの割合の確認

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
n = 100000
selected = rng.random(n) < 0.15                    # 15% のトークンを選ぶ
kind = rng.random(n)
to_mask = selected & (kind < 0.8)                  # 選んだうち 80% は [MASK]
to_rand = selected & (kind >= 0.8) & (kind < 0.9)  # 10% はランダムな別の語
to_keep = selected & (kind >= 0.9)                 # 10% はそのまま
print(round(selected.mean(), 3))
print(round(to_mask.sum() / selected.sum(), 2), round(to_rand.sum() / selected.sum(), 2),
      round(to_keep.sum() / selected.sum(), 2))

**本書に載っている出力**
```
0.152
0.8 0.1 0.1
```

## リスト5.27　BERT-base と GPT-2 small の設計

In [ ]:
from transformers import BertConfig, BertModel, GPT2Config, GPT2Model

bert = BertModel(BertConfig())               # BERT-base と同じ設計（重みは乱数）
gpt2 = GPT2Model(GPT2Config())               # GPT-2 small と同じ設計
cb, cg = bert.config, gpt2.config
print(cb.num_hidden_layers, cb.hidden_size, cb.num_attention_heads,
      sum(p.numel() for p in bert.parameters()))
print(cg.n_layer, cg.n_embd, cg.n_head,
      sum(p.numel() for p in gpt2.parameters()))

**本書に載っている出力**
```
12 768 12 109482240
12 768 12 124439808
```